# Inference with Original Models

This notebook provides simplified inference code for the models implemented under `original_models/` it covers:
- Reconstruction from the FREUD model
- Forecasting with the RaMViD Latent Space Model (LSM)

## 1. FREUD Reconstructions

In [1]:
from data.sevir import SevirDataModule

dmodule = SevirDataModule(
    sevir_pth = "/export/group/datasets/SEVIR",  # TODO: change to local data path
    seq_len = 25,
    normalize = True,
    batch_size = 1,
    num_workers = 8,
    val_files_index = "data/test_data.txt",
    use_duplicate_validation_seq = True,
    seed = 42,
)

train_loader = dmodule.train_dataloader()
val_loader = dmodule.val_dataloader()

for batch in val_loader:
    print(batch.keys())
    break

dict_keys(['x', 'metadata'])


In [10]:
import torch
import mediapy as mp
import einops
from jaxtyping import Float
from original_model.freud import FreudDiffusionAE

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

def show_video(x, title=None):
    vid = x.float().cpu().numpy()
    vid = einops.repeat(vid, "t h w 1 -> t h w (1 three)", three=3)
    vid = (vid + 1) * 127.5
    vid = vid.astype('uint8')
    mp.show_video(vid, fps=5, title=title)

@torch.inference_mode()
@torch.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu', dtype=torch.bfloat16)
def get_recon(
    model: FreudDiffusionAE,
    x: Float[torch.Tensor, "B T H W C"],
    decoding_sample_steps: int = 10,
):
    model.eval()
    x = x.to(DEVICE).to(torch.bfloat16)
    x = einops.rearrange(x, "b t h w c -> b c t h w")
    encoded = model.encode(x)

    noise = torch.randn_like(x)

    decoded = model.decode(encoded, noise, sample_steps=decoding_sample_steps).detach()
    decoded = einops.rearrange(decoded, "b c t h w -> b t h w c")
    return decoded, encoded

In [4]:
from original_model.freud import load_freud_dit_small_p4_rain

model = load_freud_dit_small_p4_rain(
    checkpoint_path="ckpt/original_freud.pt"
)
model = model.to(device=DEVICE, dtype=torch.bfloat16)
model.eval()

/export/home/ra37rok/Projects/weather-public/original_model/freud.py:964: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(checkpoint_path, map_location

FreudDiffusionAE(
  (unet): Transformer(
    (merges): ModuleList(
      (0): TokenMerge3D(
        (proj): Linear(in_features=16, out_features=96, bias=False)
      )
    )
    (down_levels): ModuleList(
      (0): ModuleList(
        (0-1): 2 x GenericTransformerLayer(
          (self_attn): FactorizedAttentionBlock(
            d_head=32,
            (norm): AdaRMSNorm(
              eps=1e-06,
              (linear): Linear(in_features=384, out_features=96, bias=False)
            )
            (qkv_proj): Linear(in_features=96, out_features=288, bias=False)
            (pos_emb): AxialRoPE3D()
            (out_proj): Linear(in_features=96, out_features=96, bias=False)
            (spatial_qkv_proj): Linear(in_features=96, out_features=288, bias=False)
            (spatial_norm): AdaRMSNorm(
              eps=1e-06,
              (linear): Linear(in_features=384, out_features=96, bias=False)
            )
            (spatial_out_proj): Linear(in_features=96, out_features=96, bias=

In [11]:
x = batch['x'].to(device=DEVICE, dtype=DTYPE)
recon, latent = get_recon(model, x)
print(f"Input x shape: {x.shape}, recon shape: {recon.shape}, latent shape: {latent.shape}")

recon_ = recon.float().clamp(-1, 1)

to_show = torch.cat([x, recon_, recon_ - x], dim=3)
show_video(to_show[0], title="Input | Recon | Diff")

Input x shape: torch.Size([1, 25, 384, 384, 1]), recon shape: torch.Size([1, 25, 384, 384, 1]), latent shape: torch.Size([1, 4, 25, 48, 48])


Input | Recon | Diff This browser does not support the video tag.


## 2. LSM Forecasts

In [1]:
from data.sevir import SevirDataModule

dmodule = SevirDataModule(
    sevir_pth = "/export/group/datasets/SEVIR",  # TODO: change to local data path
    seq_len = 25,
    normalize = True,
    batch_size = 1,
    num_workers = 8,
    val_files_index = "data/test_data.txt",
    use_duplicate_validation_seq = True,
    seed = 42,
)

train_loader = dmodule.train_dataloader()
val_loader = dmodule.val_dataloader()

for batch in val_loader:
    print(batch.keys())
    break

dict_keys(['x', 'metadata'])


In [ ]:
import torch
import mediapy as mp
import einops
from jaxtyping import Float
from original_model.lsm import LatentRF3DRamvid

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

def show_video(x, title=None):
    vid = x.float().cpu().numpy()
    vid = einops.repeat(vid, "t h w 1 -> t h w (1 three)", three=3)
    vid = (vid + 1) * 127.5
    vid = vid.astype('uint8')
    mp.show_video(vid, fps=5, title=title)

@torch.inference_mode()
@torch.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu', dtype=torch.bfloat16)
def get_pred(
    model: LatentRF3DRamvid,
    sample: Float[torch.Tensor, "B T H W C"],
    cond_indices: Float[torch.Tensor, "B T"],
    sample_steps: int = 25,
    decoder_sample_steps: int = 10,
    device: torch.device = torch.device("cuda:0"),
    dtype: torch.dtype = torch.bfloat16,
    decode_chunk_size: int = 4,
) -> Float[torch.Tensor, "B T H W C"]:
    cond_indices = cond_indices.bool().to(device)
    sample = sample.to(device)

    if sample.shape[1] == cond_indices.shape[1]:
        # Input follows B T H W C layout.
        sample = einops.rearrange(sample, "b t h w c -> b c t h w")
    elif sample.shape[2] != cond_indices.shape[1]:
        raise ValueError(
            f"Could not align cond_indices with sample time axis: {sample.shape=}, {cond_indices.shape=}"
        )

    conditional_idx = [torch.where(cond_indices[i])[0].tolist() for i in range(cond_indices.size(0))]

    with torch.autocast(device_type=device.type, dtype=dtype):
        sample = sample.to(dtype)
        encoded = model._encode(sample)
        noise = model._get_noise(torch.ones_like(encoded))

        for i, indices in enumerate(conditional_idx):
            if len(indices) > 0:
                noise[i, :, indices, :, :] = encoded[i, :, indices, :, :]

        preds = []
        for start in range(0, noise.size(0), decode_chunk_size):
            end = min(start + decode_chunk_size, noise.size(0))
            preds.append(
                model.sample(
                    z=noise[start:end],
                    conditional_idx=conditional_idx[start:end],
                    sample_steps=sample_steps,
                    decode_sample_steps=decoder_sample_steps,
                )
            )
        pred = torch.cat(preds, dim=0)

    pred = pred.float().clamp(-1, 1)
    pred = einops.rearrange(pred, "b c t h w -> b t h w c")
    return pred

In [3]:
from original_model.lsm import load_rf_ramvid_dit_L

model = load_rf_ramvid_dit_L(
    checkpoint_path="ckpt/original_lsm.pt",
    first_stage_checkpoint_path="ckpt/original_freud.pt"
)
model = model.to(device=DEVICE, dtype=torch.bfloat16)
model.eval()

/export/home/ra37rok/Projects/weather-public/original_model/freud.py:964: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(checkpoint_path, map_location

LatentRF3DRamvid(
  (unet): Transformer(
    (merges): ModuleList()
    (down_levels): ModuleList()
    (splits): ModuleList()
    (up_levels): ModuleList()
    (mid_merge): TokenMerge3D(
      (proj): Linear(in_features=16, out_features=1024, bias=False)
    )
    (mid_level): ModuleList(
      (0-23): 24 x GenericTransformerLayer(
        (self_attn): FactorizedAttentionBlock(
          d_head=64,
          (norm): AdaRMSNorm(
            eps=1e-06,
            (linear): Linear(in_features=512, out_features=1024, bias=False)
          )
          (qkv_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (pos_emb): AxialRoPE3D()
          (out_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (spatial_qkv_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (spatial_norm): AdaRMSNorm(
            eps=1e-06,
            (linear): Linear(in_features=512, out_features=1024, bias=False)
          )
          (spatial_out_proj)

In [4]:
INPUT_LENGTH = 12

x = batch['x']
sample = x.clone().to(device=DEVICE, dtype=DTYPE)
noise = torch.randn_like(sample)
sample[:, INPUT_LENGTH:] = noise[:, INPUT_LENGTH:]
cond_indices = torch.zeros(sample.shape[0], sample.shape[1], dtype=torch.bool, device=sample.device)
cond_indices[:, :INPUT_LENGTH] = True
pred = get_pred(model, sample, cond_indices, sample_steps=25, decoder_sample_steps=10, device=DEVICE, dtype=DTYPE)
print(f"Sample shape: {sample.shape}, pred shape: {pred.shape}")

[DEBUG] sample shape: torch.Size([1, 1, 25, 384, 384]), cond_indices shape: torch.Size([1, 25])
[DEBUG] noise shape: torch.Size([1, 4, 25, 48, 48])
Sample shape: torch.Size([1, 25, 384, 384, 1]), pred shape: torch.Size([1, 25, 384, 384, 1])


In [5]:
pred_ = pred.float().cpu().clamp(-1, 1)

to_show = torch.cat([x, pred_, pred_ - x], dim=3)
show_video(to_show[0], title="Input | Prediction | Diff")

Input | Prediction | Diff This browser does not support the video tag.
